# 04 Water Quality Context

**Series:** Pine Ridge Hydrology  
**Author:** Lilly Jones, PhD                             
**Primary Focus:** Pine Ridge Reservation/Oglala Sioux Tribe  
**Collective:** Oglala Lakota  
**Data Sources:** USGS/EPA Water Quality Portal public observations

## Water Quality on Oglala Lakota Lands
Drinking water quality on Pine Ridge reflects both natural
geochemistry and the legacy of infrastructure underinvestment. Key concerns:

- **Naturally occurring mineralization** the Arikaree and Ogallala
  aquifers in this region can carry elevated TDS (total dissolved solids),
  sulfate, and fluoride from the geologic formations
- **Nitrate** from agricultural activity in upgradient areas and from
  septic systems in communities without central wastewater infrastructure
- **Variable quality by source** spring, well, and surface water sources
  show wide variation; some sources are marginal or impaired

## Data Approach
Public water quality data from the USGS/EPA Water Quality Portal is sparse
in the configured query. This is a limitation of the selected public record, not evidence that water, knowledge, local monitoring, or community concern is absent. This public teaching path does not load OST-controlled data.

## Research Questions
- Which parameters exceed EPA MCLs in the public record?
- What does the spatial distribution of exceedances look like?
- Where does the selected public record have limited spatial, temporal, or parameter coverage?
- What additional evidence or local expertise would be needed to evaluate conditions not represented by the public record?

## Learning Objectives

By the end of this notebook, learners will be able to:

- inspect sampling coverage and harmonize parameter units before comparison
- distinguish a screening flag from a regulatory, exposure, or health conclusion
- identify additional evidence needed to investigate a possible contaminant source

## Prerequisites and Timing

Allow approximately 75–100 minutes. Before beginning, activate the repository environment, read the series governance statement, and complete the preceding notebook where applicable. Work in pairs and rotate analyst, data-steward, skeptic, and documentarian roles.

## Governance Checkpoint

This notebook uses public environmental data describing Oglala Lakota lands and waters. Public availability does not establish permission for every reuse or interpretation. Do not add OST-controlled data, sensitive locations, or community knowledge. Results are educational and screening-level pending OLC/OST review.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime
import requests
from io import StringIO

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as ctx
import yaml

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED,
    OUTPUTS_DIR, FIGURES_DIR,
    REPO_ROOT as _REPO_ROOT,
)
from src.config import load_config, streamflow_site_ids, streamflow_site_names

CONFIG = load_config()
STUDY_BBOX = tuple(CONFIG["study_area"]["hydrologic_context_bbox"])
STUDY_NAMES = [CONFIG["study_area"]["people"]]
STUDY_CENTROIDS = {CONFIG["study_area"]["people"]: CONFIG["study_area"]["centroid"]}
PINE_RIDGE_STREAMGAGES = {
    site["name"]: str(site["id"]) for site in CONFIG["usgs_streamflow_sites"]
}

from src.loaders import (
    load_tribal_boundaries,
    load_water_quality,
)
from src.indicators import flag_water_quality_exceedances
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

with open(_REPO_ROOT/"config"/"config.yaml") as f:
    CONFIG = yaml.safe_load(f)

WQ_THRESHOLDS = CONFIG["thresholds"]["water_quality"]
START_YEAR    = CONFIG["analysis"]["start_year"]

print("Water quality thresholds (EPA MCLs):")
for k, v in WQ_THRESHOLDS.items():
    print(f"  {k}: {v}")

In [ ]:
# Print data sovereignty notice at the top of every notebook 
print_data_acknowledgment(
    source_keys=["usgs_nwis_water_quality", "tribal_water_quality", "census_aiannh"]
)

## Load Boundaries

In [ ]:
boundaries_path = OUTPUTS_DIR/"pine_ridge_census_boundary.geojson"
if boundaries_path.exists():
    study_boundary = gpd.read_file(boundaries_path)
else:
    study_boundary = load_tribal_boundaries()

primary = study_boundary[study_boundary["common_name"].isin(STUDY_NAMES)]
print(f"Boundary features: {len(study_boundary)} | Selected study feature: {len(primary)}")

## Load Public Water Quality Data (USGS/EPA WQP)

In [ ]:
# Query WQP for key drinking water parameters
# Split into Pine Ridge bboxes (WQP rejects large bounding boxes)
from datetime import datetime

WQ_PARAMS = [
    "Nitrate",
    "pH",
    "Total dissolved solids",
    "Turbidity",
    "Arsenic",
    "Fluoride",
]

# WQP requires MM-dd-yyyy format
start_wqp = datetime.strptime(f"{START_YEAR}-01-01", "%Y-%m-%d").strftime("%m-%d-%Y")

wq_parts = []
for name, bbox in [("Pine Ridge", STUDY_BBOX)]:
    print(f"Querying WQP for {name}...")
    df = load_water_quality(
        bbox            = bbox,
        start_date      = f"{START_YEAR}-01-01",
        characteristics = WQ_PARAMS,
    )
    if not df.empty:
        df["region"] = name
        wq_parts.append(df)
        print(f"  {len(df):,} records")
    else:
        print(f"  No data returned: monitoring gap finding for {name}")

if wq_parts:
    wq_public = pd.concat(wq_parts, ignore_index=True)
    print(f"\nTotal public WQ records: {len(wq_public):,}")
    if "CharacteristicName" in wq_public.columns:
        print("\nBy parameter:")
        print(wq_public["CharacteristicName"].value_counts().to_string())
    if "ActivityStartDate" in wq_public.columns:
        dates = pd.to_datetime(wq_public["ActivityStartDate"], errors="coerce")
        print(f"\nDate range: {dates.min().date()} to {dates.max().date()}")
else:
    wq_public = pd.DataFrame()
    print("\nNo public WQP data returned for either reservation.")
    print("Sparse public WQ monitoring on Tribal lands is a federal")
    print("infrastructure equity gap, not evidence of clean water.")
    print("The selected public record has limited coverage; causes and implications require additional evidence.")

In [ ]:
# Fetch station coordinates from WQP station service (separate from results)
def fetch_wqp_stations(bbox):
    from datetime import datetime
    r = requests.get(
        "https://www.waterqualitydata.us/data/Station/search/",
        params={
            "bBox":     f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
            "mimeType": "csv",
        },
        timeout=120,
    )
    if r.status_code == 200 and len(r.content) > 200:
        return pd.read_csv(StringIO(r.text), dtype=str, low_memory=False)
    print(f"  Station fetch failed: {r.status_code} {r.headers.get('Warning','')}")
    return pd.DataFrame()

print("Fetching WQP station locations...")
stations = fetch_wqp_stations(STUDY_BBOX)

print(f"Unique stations: {len(stations)}")
if not stations.empty:
    lat_lon_cols = [c for c in stations.columns
                    if "lat" in c.lower() or "lon" in c.lower()]
    print(f"Coordinate columns: {lat_lon_cols}")

In [ ]:
# Pivot public WQ data to wide format for threshold comparison
# One row per sample, one column per parameter

if not wq_public.empty:
    # Map WQP characteristic names to our standard column names
    CHAR_MAP = {
        "Nitrate":                  "nitrate_mgl",
        "pH":                       "ph",
        "Total dissolved solids":   "tds_mgl",
        "Turbidity":                "turbidity_ntu",
        "Arsenic":                  "arsenic_ugl",
        "Fluoride":                 "fluoride_mgl",
    }

    wq_wide_parts = []
    for char, col_name in CHAR_MAP.items():
        subset = wq_public[
            wq_public.get("CharacteristicName", pd.Series()).str.lower()
            == char.lower()
        ].copy() if "CharacteristicName" in wq_public.columns else pd.DataFrame()

        if subset.empty:
            continue

        cols = {"ActivityStartDate": "date", "result_value": col_name}
        for extra in ["MonitoringLocationIdentifier", "LatitudeMeasure",
                      "LongitudeMeasure"]:
            if extra in subset.columns:
                cols[extra] = extra

        sub = subset[list(cols.keys())].rename(columns=cols)
        wq_wide_parts.append(sub)

    if wq_wide_parts:
        from functools import reduce
        wq_wide = reduce(
            lambda a, b: pd.merge(a, b, on=["date", "MonitoringLocationIdentifier"],
                                  how="outer"),
            wq_wide_parts
        )
        print(f"Wide-format WQ table: {len(wq_wide)} samples, "
              f"{len(wq_wide.columns)} columns")
    else:
        wq_wide = pd.DataFrame()
        print("Could not pivot WQ data, structure may differ from expected.")
else:
    wq_wide = pd.DataFrame()

## Threshold Exceedance Analysis

In [ ]:
# Apply threshold flags to whichever datasets are available
thresholds = {
    "nitrate_mgl":    WQ_THRESHOLDS.get("nitrate_mgl", 10.0),
    "arsenic_ugl":    WQ_THRESHOLDS.get("arsenic_ugl", 10.0),
    "tds_mgl":        WQ_THRESHOLDS.get("tds_mgl", 500.0),
    "ph":             None,
    "turbidity_ntu":  WQ_THRESHOLDS.get("turbidity_ntu", 1.0),
    "fluoride_mgl":   WQ_THRESHOLDS.get("fluoride_mgl", 4.0),
}

# Public data
if not wq_wide.empty:
    wq_wide_flagged = flag_water_quality_exceedances(wq_wide, thresholds)
    alert_cols  = [c for c in wq_wide_flagged.columns if c.endswith("_alert")]
    print("PUBLIC WQ EXCEEDANCES (EPA thresholds)")
    for col in alert_cols:
        param = col.replace("_alert", "")
        n     = wq_wide_flagged[col].sum()
        total = wq_wide_flagged[col].notna().sum()
        if total > 0:
            print(f"  {param}: {n}/{total} samples exceeded threshold "
                  f"({n/total*100:.1f}%)")
    if "any_alert" in wq_wide_flagged.columns:
        any_exc = wq_wide_flagged["any_alert"].sum()
        print(f"\n  Samples with ANY exceedance: {any_exc}/{len(wq_wide_flagged)}")
else:
    wq_wide_flagged = pd.DataFrame()
    print("No public WQ data available for threshold analysis.")


## Visualizations

In [ ]:
# Exceedance summary bar chart
datasets = []
if not wq_wide_flagged.empty:
    datasets.append(("Public (WQP)", wq_wide_flagged, "#1A5276"))

params = ["nitrate_mgl", "arsenic_ugl", "tds_mgl",
          "turbidity_ntu", "fluoride_mgl", "ph"]
param_labels = ["Nitrate (MCL: 10 mg/L)", "Arsenic (MCL: 10 µg/L)",
                "TDS (sec: 500 mg/L)", "Turbidity (1 NTU)",
                "Fluoride (MCL: 4 mg/L)", "pH (6.5–8.5)"]

if datasets:
    fig, ax = plt.subplots(figsize=(11, 6))
    width   = 0.35
    x       = np.arange(len(params))

    for i, (label, df, color) in enumerate(datasets):
        pcts = []
        for p in params:
            col = f"{p}_alert"
            if col in df.columns:
                total = df[col].notna().sum()
                pct   = df[col].sum() / total * 100 if total > 0 else 0
            else:
                pct = 0
            pcts.append(pct)
        offset = width * (i - len(datasets) / 2 + 0.5)
        ax.bar(x + offset, pcts, width, color=color, alpha=0.8, label=label)

    ax.set_xticks(x)
    ax.set_xticklabels(param_labels, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("% of samples exceeding threshold", fontsize=10)
    ax.set_title(
        "Water Quality Exceedances for the Pine Ridge Hydrologic Context\n"
        "Compared to EPA Maximum Contaminant Levels (MCLs)",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9)
    despine(ax)
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"04_wq_exceedances.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()
else:
    print("No water quality data available for visualization.")
    print()
    print("The configured public query returned limited coverage.")
    print("Do not infer that water, local knowledge, or other monitoring is absent.")
    print("Causes and implications require additional evidence and review.")

In [ ]:
# Plot WQ site locations 

fig, ax = plt.subplots(figsize=(12, 8))

study_boundary.to_crs(3857).plot(
    ax=ax, facecolor="#F4F6F7", edgecolor="#566573",
    linewidth=1.5, zorder=1,
)
primary.to_crs(3857).plot(
    ax=ax, facecolor="#D6EAF8", edgecolor="#2471A3",
    linewidth=2, zorder=2, alpha=0.5,
)

for _, nation in primary.iterrows():
    c = nation.geometry.centroid
    c3857 = gpd.GeoSeries([c], crs=CRS_GEOGRAPHIC).to_crs(3857).iloc[0]
    ax.annotate(
        nation["common_name"].replace(" (", "\n("),
        (c3857.x, c3857.y), ha="center", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7),
    )

if not stations.empty:
    stations["LatitudeMeasure"]  = pd.to_numeric(stations["LatitudeMeasure"],  errors="coerce")
    stations["LongitudeMeasure"] = pd.to_numeric(stations["LongitudeMeasure"], errors="coerce")
    stations_clean = stations.dropna(subset=["LatitudeMeasure", "LongitudeMeasure"])

    # Filter to the configured Pine Ridge hydrologic context
    in_study_area = (
        (stations_clean["LongitudeMeasure"] >= STUDY_BBOX[0]) &
        (stations_clean["LongitudeMeasure"] <= STUDY_BBOX[2]) &
        (stations_clean["LatitudeMeasure"]  >= STUDY_BBOX[1]) &
        (stations_clean["LatitudeMeasure"]  <= STUDY_BBOX[3])
    )
    stations_clean = stations_clean[in_study_area].copy()

    wq_gdf = gpd.GeoDataFrame(
        stations_clean,
        geometry=gpd.points_from_xy(
            stations_clean["LongitudeMeasure"],
            stations_clean["LatitudeMeasure"],
        ),
        crs=CRS_GEOGRAPHIC,
    )
    wq_gdf.to_crs(3857).plot(
        ax=ax, color="#1A5276", marker="o", markersize=3,
        alpha=0.5, zorder=3,
        label=f"WQP monitoring stations (n={len(wq_gdf):,})",
    )
# Clip map extent to the configured Pine Ridge context bbox
    combined_bbox = STUDY_BBOX
    # Convert bbox corners to Web Mercator for ax.set_xlim/ylim
    bbox_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(
            [combined_bbox[0], combined_bbox[2]],
            [combined_bbox[1], combined_bbox[3]],
        ),
        crs=CRS_GEOGRAPHIC,
    ).to_crs(3857)
    xs = bbox_gdf.geometry.x.values
    ys = bbox_gdf.geometry.y.values
    margin = 20_000   # 20 km margin in meters
    ax.set_xlim(xs[0] - margin, xs[1] + margin)
    ax.set_ylim(ys[0] - margin, ys[1] + margin)
try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.4)
except Exception:
    pass

ax.set_axis_off()
ax.legend(fontsize=9, loc="lower right")
ax.set_title(
    "Water Quality Monitoring Coverage for Pine Ridge\n"
    f"Public WQP stations (n={len(wq_gdf):,}) | "
    f"22,018 result records | 2000–2024",
    fontsize=10, fontweight="bold",
)
plt.tight_layout()
try:
    fig.savefig(FIGURES_DIR/"04_wq_monitoring_coverage.png",
                dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

print(f"Mapped {len(wq_gdf):,} stations across Pine Ridge study areas")

## Exports

In [ ]:
if not wq_wide_flagged.empty:
    wq_wide_flagged.to_csv(OUTPUTS_DIR/"wq_public_flagged.csv", index=False)
    print("Exported to outputs/wq_public_flagged.csv")



In [ ]:
print(generate_citations(["usgs_nwis_water_quality"]))

## Learner Checkpoint

Choose one parameter and verify its reported unit, screening benchmark source, sample period, and number of sites before interpreting a flag.

## Interpretation Protocol

Before writing a conclusion, separate:

1. **Observation:** what the computed public data show, including unit, period, spatial scope, and missingness.
2. **Interpretation:** a plausible explanation, stated with uncertainty.
3. **Additional evidence:** literature, local monitoring, expertise, or validation needed to evaluate that explanation.
4. **Decision authority:** who is authorized to approve publication, thresholds, or management action.

Do not convert monitoring absence, association, a screening flag, or scenario output into a causal, regulatory, health, policy, or community conclusion.

## Contribution Activity

Improve one unit conversion note, benchmark citation, coverage statement, or limitation. Do not infer a contaminant source from a flag alone. Review the change with a partner and record what became clearer or more defensible.

## Evidence Record and Next Step

Record one regenerated result, its source and scope, one transformation, one limitation, and one question requiring more evidence or local knowledge.

Notebook 05 uses NOAA Climate Division 7 as a regional proxy for historical drought context.